# Prepare MELD For Test

This notebook materializes MELD into a flat test/evaluation bundle:

- all resolvable `.wav` files are copied into `data/MELD/meld_for_test/`
- one metadata file is written to `data/MELD/metadata.csv`
- metadata columns follow the requested `final_cols` order

Default source is `meld_vi_text_original_audio_all.csv`, filtered to rows whose `.wav` file exists. With the current dataset this should produce 11,686 audio files and 11,686 metadata rows.

In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path
import os
import shutil
import wave

import pandas as pd
from IPython.display import display


## 1. Config

In [2]:
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_ROOT = PROJECT_ROOT / 'data' / 'MELD'
SOURCE_AUDIO_ROOT = DATA_ROOT / 'original_meld_audio_wav'
OUT_AUDIO_DIR = DATA_ROOT / 'meld_for_test'
OUT_METADATA_PATH = DATA_ROOT / 'metadata.csv'

# Use all metadata rows, then keep only rows that have a resolvable wav file.
# If you want only the clean benchmark subset, switch to:
# SOURCE_CSV_NAME = 'meld_vi_text_original_audio_clean.csv'
SOURCE_CSV_NAME = 'meld_vi_text_original_audio_all.csv'

# copy: physically copy wav files.
# hardlink: save disk space on the same filesystem.
# symlink: create links; may require privileges on Windows.
COPY_MODE = 'copy'
OVERWRITE_EXISTING = False
CLEAN_OUTPUT_DIR_BEFORE_COPY = False

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('DATA_ROOT         =', DATA_ROOT)
print('SOURCE_AUDIO_ROOT =', SOURCE_AUDIO_ROOT)
print('OUT_AUDIO_DIR     =', OUT_AUDIO_DIR)
print('OUT_METADATA_PATH =', OUT_METADATA_PATH)

assert DATA_ROOT.exists(), f'Missing data root: {DATA_ROOT}'
assert SOURCE_AUDIO_ROOT.exists(), f'Missing audio root: {SOURCE_AUDIO_ROOT}'
assert COPY_MODE in {'copy', 'hardlink', 'symlink'}


PROJECT_ROOT      = /home/emotalk/mer2
DATA_ROOT         = /home/emotalk/mer2/data/MELD
SOURCE_AUDIO_ROOT = /home/emotalk/mer2/data/MELD/original_meld_audio_wav
OUT_AUDIO_DIR     = /home/emotalk/mer2/data/MELD/meld_for_test
OUT_METADATA_PATH = /home/emotalk/mer2/data/MELD/metadata.csv


## 2. Requested Metadata Columns

In [3]:
final_cols = [
    'sample_id',
    'utterance_id',
    'split',
    'label',
    'text',
    'raw_text',
    'normalized_text',
    'transcript_final',
    'text_for_model',
    'audio_path',
    'audio_relpath',
    'group_id',
    'speaker',
    'source',
    'text_en',
    'text_vi',
    'duration_sec',
    'sample_rate',
    'original_audio_source',
]

print('Number of final columns:', len(final_cols))
display(pd.DataFrame({'final_cols': final_cols}))


Number of final columns: 19


,final_cols
0,sample_id
1,utterance_id
2,split
3,label
4,text
5,raw_text
6,normalized_text
7,transcript_final
8,text_for_model
9,audio_path


## 3. Load Source Metadata

In [4]:
source_csv_path = DATA_ROOT / SOURCE_CSV_NAME
assert source_csv_path.exists(), f'Missing source csv: {source_csv_path}'

source_df = pd.read_csv(source_csv_path)
missing_cols = [col for col in final_cols if col not in source_df.columns]
assert not missing_cols, f'Missing required columns in {SOURCE_CSV_NAME}: {missing_cols}'

print('Source CSV:', source_csv_path)
print('Rows:', len(source_df))
print('Columns:', len(source_df.columns))
display(source_df[final_cols].head(3).T)


Source CSV: /home/emotalk/mer2/data/MELD/meld_vi_text_original_audio_all.csv
Rows: 11687
Columns: 29


,0,1,2
sample_id,meld_test_d0000_u001,meld_test_d0000_u002,meld_test_d0001_u000
utterance_id,meld_test_d0000_u001,meld_test_d0000_u002,meld_test_d0001_u000
split,test,test,test
label,anger,neutral,neutral
text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
raw_text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
normalized_text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
transcript_final,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
text_for_model,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
audio_path,original_meld_audio_wav/meld_test_d0000_u001.wav,original_meld_audio_wav/meld_test_d0000_u002.wav,original_meld_audio_wav/meld_test_d0001_u000.wav


## 4. Index All Source WAV Files

In [5]:
source_wav_paths = sorted(SOURCE_AUDIO_ROOT.rglob('*.wav'))
source_audio_index: dict[str, Path] = {}
duplicate_wav_names = []

for wav_path in source_wav_paths:
    if wav_path.name in source_audio_index:
        duplicate_wav_names.append(wav_path.name)
    source_audio_index[wav_path.name] = wav_path.resolve()

print('Source wav files:', len(source_wav_paths))
print('Unique wav basenames:', len(source_audio_index))
print('Duplicate basenames:', len(duplicate_wav_names))
assert not duplicate_wav_names, 'Duplicate wav basenames found; basename-based resolution is unsafe.'

shard_rows = []
for shard_dir in sorted(SOURCE_AUDIO_ROOT.glob('original_meld_audio_wav_shard_*')):
    if shard_dir.is_dir():
        shard_rows.append({
            'shard': shard_dir.name,
            'wav_files': len(list(shard_dir.rglob('*.wav'))),
        })
display(pd.DataFrame(shard_rows))


Source wav files: 11686
Unique wav basenames: 11686
Duplicate basenames: 0


,shard,wav_files
0,original_meld_audio_wav_shard_0000,1000
1,original_meld_audio_wav_shard_0001,1000
2,original_meld_audio_wav_shard_0002,1000
3,original_meld_audio_wav_shard_0003,1000
4,original_meld_audio_wav_shard_0004,1000
5,original_meld_audio_wav_shard_0005,1000
6,original_meld_audio_wav_shard_0006,1000
7,original_meld_audio_wav_shard_0007,1000
8,original_meld_audio_wav_shard_0008,1000
9,original_meld_audio_wav_shard_0009,1000


## 5. Resolve Metadata Rows To Existing WAV Files

In [6]:
def _clean_string(value) -> str:
    if pd.isna(value):
        return ''
    return str(value).strip()


def expected_wav_name(row: pd.Series) -> str:
    for col in ['audio_path', 'audio_relpath']:
        value = _clean_string(row.get(col))
        if value:
            return Path(value).name
    return f"{_clean_string(row['sample_id'])}.wav"


resolved_df = source_df.copy()
resolved_df['expected_wav_name'] = resolved_df.apply(expected_wav_name, axis=1)
resolved_df['source_audio_path'] = resolved_df['expected_wav_name'].map(source_audio_index)
resolved_df['audio_exists'] = resolved_df['source_audio_path'].notna()

resolve_summary = pd.DataFrame([
    {'metric': 'metadata_rows', 'value': len(resolved_df)},
    {'metric': 'source_wav_files', 'value': len(source_wav_paths)},
    {'metric': 'rows_with_audio', 'value': int(resolved_df['audio_exists'].sum())},
    {'metric': 'rows_missing_audio', 'value': int((~resolved_df['audio_exists']).sum())},
])
display(resolve_summary)

missing_audio_df = resolved_df.loc[
    ~resolved_df['audio_exists'],
    ['sample_id', 'utterance_id', 'split', 'label', 'audio_path', 'audio_relpath', 'expected_wav_name'],
]
display(missing_audio_df.head(20))

bundle_df = resolved_df.loc[resolved_df['audio_exists']].copy().reset_index(drop=True)
if SOURCE_CSV_NAME == 'meld_vi_text_original_audio_all.csv':
    assert len(bundle_df) == len(source_wav_paths), (
        'Number of metadata rows with audio must equal number of source wav files. '
        f"rows_with_audio={len(bundle_df)}, source_wav_files={len(source_wav_paths)}"
    )
else:
    print('Subset source CSV selected; metadata rows may be fewer than total source wav files.')


,metric,value
0,metadata_rows,11687
1,source_wav_files,11686
2,rows_with_audio,11686
3,rows_missing_audio,1


,sample_id,utterance_id,split,label,audio_path,audio_relpath,expected_wav_name
3249,meld_train_d0125_u003,meld_train_d0125_u003,train,neutral,NaN,NaN,meld_train_d0125_u003.wav


## 6. Build Final Metadata For The Flat Folder

In [7]:
data_root_rel = DATA_ROOT.relative_to(PROJECT_ROOT).as_posix()

bundle_df['target_wav_name'] = bundle_df['expected_wav_name']
bundle_df['target_audio_path'] = bundle_df['target_wav_name'].map(lambda name: OUT_AUDIO_DIR / name)

# audio_path is project-root relative, useful for repo training/eval code.
# audio_relpath is DATA_ROOT relative, useful when metadata.csv is consumed from data/MELD.
bundle_df['audio_relpath'] = bundle_df['target_wav_name'].map(lambda name: f'{OUT_AUDIO_DIR.name}/{name}')
bundle_df['audio_path'] = bundle_df['audio_relpath'].map(lambda rel: f'{data_root_rel}/{rel}')

metadata_df = bundle_df[final_cols].copy()

assert list(metadata_df.columns) == final_cols
assert metadata_df['audio_path'].str.startswith('data/MELD/meld_for_test/').all()
assert metadata_df['audio_relpath'].str.startswith('meld_for_test/').all()
assert metadata_df['sample_id'].is_unique, 'sample_id must be unique.'
assert metadata_df['audio_relpath'].is_unique, 'audio_relpath must be unique.'

display(metadata_df.head(3).T)
display(pd.crosstab(metadata_df['split'], metadata_df['label']))
print('Prepared metadata rows:', len(metadata_df))


,0,1,2
sample_id,meld_test_d0000_u001,meld_test_d0000_u002,meld_test_d0001_u000
utterance_id,meld_test_d0000_u001,meld_test_d0000_u002,meld_test_d0001_u000
split,test,test,test
label,anger,neutral,neutral
text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
raw_text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
normalized_text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
transcript_final,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
text_for_model,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
audio_path,data/MELD/meld_for_test/meld_test_d0000_u001.wav,data/MELD/meld_for_test/meld_test_d0000_u002.wav,data/MELD/meld_for_test/meld_test_d0001_u000.wav


label,anger,fear,happiness,neutral,sadness
split,,,,,
test,344,50,401,1253,208
train,1107,268,1739,4701,680
valid,153,40,163,469,110


Prepared metadata rows: 11686


## 7. Copy WAV Files Into `meld_for_test`

In [8]:
def materialize_wav(src: Path, dst: Path) -> str:
    src = Path(src)
    dst = Path(dst)

    if dst.exists():
        if not OVERWRITE_EXISTING and dst.stat().st_size == src.stat().st_size:
            return 'skipped_existing'
        if OVERWRITE_EXISTING:
            dst.unlink()
        else:
            return 'skipped_name_conflict'

    if COPY_MODE == 'copy':
        shutil.copy2(src, dst)
        return 'copied'
    if COPY_MODE == 'hardlink':
        os.link(src, dst)
        return 'hardlinked'
    if COPY_MODE == 'symlink':
        os.symlink(src, dst)
        return 'symlinked'
    raise ValueError(f'Unknown COPY_MODE: {COPY_MODE}')


OUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)

if CLEAN_OUTPUT_DIR_BEFORE_COPY:
    for old_wav in OUT_AUDIO_DIR.glob('*.wav'):
        old_wav.unlink()

status_counts = Counter()
num_rows = len(bundle_df)

for idx, row in bundle_df.iterrows():
    status = materialize_wav(Path(row['source_audio_path']), Path(row['target_audio_path']))
    status_counts[status] += 1
    if (idx + 1) % 1000 == 0 or (idx + 1) == num_rows:
        print(f'Processed {idx + 1}/{num_rows}')

display(pd.DataFrame(sorted(status_counts.items()), columns=['status', 'count']))


Processed 1000/11686
Processed 2000/11686
Processed 3000/11686
Processed 4000/11686
Processed 5000/11686
Processed 6000/11686
Processed 7000/11686
Processed 8000/11686
Processed 9000/11686
Processed 10000/11686
Processed 11000/11686
Processed 11686/11686


,status,count
0,copied,11686


## 8. Write `metadata.csv`

In [9]:
metadata_df.to_csv(OUT_METADATA_PATH, index=False, encoding='utf-8')

print('Wrote metadata:', OUT_METADATA_PATH)
print('Rows:', len(metadata_df))
print('Columns:', len(metadata_df.columns))
display(pd.read_csv(OUT_METADATA_PATH).head(3).T)


Wrote metadata: /home/emotalk/mer2/data/MELD/metadata.csv
Rows: 11686
Columns: 19


,0,1,2
sample_id,meld_test_d0000_u001,meld_test_d0000_u002,meld_test_d0001_u000
utterance_id,meld_test_d0000_u001,meld_test_d0000_u002,meld_test_d0001_u000
split,test,test,test
label,anger,neutral,neutral
text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
raw_text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
normalized_text,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
transcript_final,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
text_for_model,Như vậy Monica có thể theo dõi. cô ấy có thể n...,Biết gì không?,"Thôi nào, Lydia, cô làm được mà."
audio_path,data/MELD/meld_for_test/meld_test_d0000_u001.wav,data/MELD/meld_for_test/meld_test_d0000_u002.wav,data/MELD/meld_for_test/meld_test_d0001_u000.wav


## 9. Validate Final Bundle

In [10]:
target_wavs = sorted(OUT_AUDIO_DIR.glob('*.wav'))
target_wav_names = {p.name for p in target_wavs}
metadata_wav_names = {Path(x).name for x in metadata_df['audio_relpath']}

missing_in_folder = sorted(metadata_wav_names - target_wav_names)
extra_in_folder = sorted(target_wav_names - metadata_wav_names)

validation_summary = pd.DataFrame([
    {'metric': 'metadata_rows', 'value': len(metadata_df)},
    {'metric': 'wav_files_in_meld_for_test', 'value': len(target_wavs)},
    {'metric': 'missing_wavs_for_metadata', 'value': len(missing_in_folder)},
    {'metric': 'extra_wavs_without_metadata', 'value': len(extra_in_folder)},
])
display(validation_summary)

assert list(pd.read_csv(OUT_METADATA_PATH, nrows=0).columns) == final_cols
assert len(target_wavs) == len(metadata_df), 'Flat wav folder count must equal metadata rows.'
assert not missing_in_folder, f'Missing wavs for metadata rows: {missing_in_folder[:10]}'
assert not extra_in_folder, f'Extra wavs without metadata rows: {extra_in_folder[:10]}'

path_checks = metadata_df['audio_path'].map(lambda p: (PROJECT_ROOT / p).exists())
relpath_checks = metadata_df['audio_relpath'].map(lambda p: (DATA_ROOT / p).exists())
assert path_checks.all(), 'Some metadata audio_path values do not exist from PROJECT_ROOT.'
assert relpath_checks.all(), 'Some metadata audio_relpath values do not exist from DATA_ROOT.'

print('Final bundle validation passed.')


,metric,value
0,metadata_rows,11686
1,wav_files_in_meld_for_test,11686
2,missing_wavs_for_metadata,0
3,extra_wavs_without_metadata,0


Final bundle validation passed.


## 10. Inspect A Few WAV Headers

In [11]:
def wav_header(path: Path) -> dict[str, object]:
    with wave.open(str(path), 'rb') as wf:
        frames = wf.getnframes()
        sample_rate = wf.getframerate()
        return {
            'file': path.name,
            'channels': wf.getnchannels(),
            'sample_rate': sample_rate,
            'sample_width_bytes': wf.getsampwidth(),
            'frames': frames,
            'duration_sec': frames / sample_rate if sample_rate else None,
        }


sample_targets = target_wavs[:3] + target_wavs[-3:]
display(pd.DataFrame([wav_header(p) for p in sample_targets]))


,file,channels,sample_rate,sample_width_bytes,frames,duration_sec
0,meld_test_d0000_u001.wav,1,24000,2,162304,6.762667
1,meld_test_d0000_u002.wav,1,24000,2,28160,1.173333
2,meld_test_d0001_u000.wav,1,24000,2,48128,2.005333
3,meld_valid_d0113_u011.wav,1,24000,2,43008,1.792000
4,meld_valid_d0113_u012.wav,1,24000,2,44032,1.834667
5,meld_valid_d0113_u013.wav,1,24000,2,36352,1.514667
